# Prediksi Beban Listrik *Day-Ahead* dan Segmentasi Profil Beban Harian
### Studi kasus: beban per jam jaringan PJM (AS): PJM East sebagai analisis utama (2002–2018), diperluas ke 9 wilayah PJM lain

**Mini Project — Artificial Intelligence untuk Teknik Elektro** &nbsp;|&nbsp; Kelompok: Alfito Putra Parindra (24/534768/TK/59290) · Muhammad Fasha (24/535041/TK/59325) · Najamudin Fawwaz Haq (24/539273/TK/59804) · Qais Faqih Basyaev (24/542181/TK/60189) · Muhammad Zeva Hardi Seswanto (24/542780/TK/60292) &nbsp;|&nbsp; Teknik Elektro, Universitas Gadjah Mada

Dua pendekatan AI pada satu data: **Clustering** (K-Means, dipilih lewat tuning *preprocessing* dan perbandingan algoritma) untuk mengelompokkan profil beban harian, dan **Regression**
(HistGradientBoosting, dibandingkan dengan Random Forest, Ridge, dan baseline sederhana) untuk memprediksi beban tiap jam **sehari ke depan**.

> **Perluasan multi-wilayah.** Analisis utama memakai PJM East (PJME). Pra-proses yang terdokumentasi dan pipeline yang sama dijalankan juga pada 9 wilayah lain (Bagian 3.2 dan 5.4); nama lengkap tiap kode wilayah tidak dijelaskan di sumber data sehingga ditulis sebagai kode saja.
>
> **Catatan.** Beban agregat wilayah jauh lebih teratur daripada konsumsi satu rumah tangga. Akurasi yang tinggi di sini terutama berasal dari sifat data
> (agregat, per jam, riwayat sangat berkorelasi), bukan dari model yang canggih (lihat Bagian 7).

**Cara menjalankan** (dari *root* repo; Python 3.10+ dan paket di `requirements.txt`): buka notebook ini lalu *Run All* (sekitar 40 menit pada 2 CPU).
Data dibaca dari `data_pjm/`, hasil tuning *clustering* penuh dari `cluster_tuning/` (dihasilkan terpisah oleh skrip di folder itu), dan semua keluaran (JSON, gambar) ditulis ke `results/`.


## 1. Problem Definition
Operator sistem tenaga harus menjadwalkan pembangkit sehari sebelumnya, sehingga membutuhkan **prediksi beban per jam untuk 24 jam ke depan**.
Selain itu, perencana perlu memahami **bentuk khas beban harian** (kapan puncak, seberapa tinggi, pada hari seperti apa).

Pertanyaan: (1) seberapa akurat beban per jam dapat diprediksi sehari ke depan hanya dari riwayat beban dan kalender, dibanding baseline sederhana?
(2) apakah hari-hari dapat dikelompokkan menjadi profil beban yang bermakna?


## 2. AI Formulation
| Tugas | Pendekatan | Input | Output | Ukuran keberhasilan |
|---|---|---|---|---|
| Segmentasi profil | **Clustering** K-Means pada profil dalam MW tanpa standardisasi (*preprocessing* dan algoritma dituning; k dari Silhouette) | profil 24 jam per hari (MW) | label *cluster* per hari | Silhouette, Davies-Bouldin, stabilitas, R² rekonstruksi, kebermaknaan *cluster* |
| Prediksi day-ahead | **Regression** HistGradientBoosting (+ RF, Ridge, baseline) | kalender + riwayat beban sampai 23:00 hari D-1 | beban (MW) tiap jam hari D | MAE, RMSE, R², **relative error** (MAE/rata-rata), dan penurunan error terhadap baseline |

Satu model regresi dipakai untuk semua jam (jam sebagai feature). Tidak ada tugas klasifikasi.


## 3. Data / Knowledge Preparation
**Sumber.** *PJME_hourly.csv*: beban listrik per jam (MW) wilayah PJM East, dari data PJM Interconnection. README repositori yang kami pakai menyatakan data berasal dari situs PJM dan tersedia di Kaggle,
dalam satuan MW, serta bahwa cakupan wilayah berubah dari tahun ke tahun. Berkas diambil dari salinan di GitHub (`panambY/Hourly_Energy_Consumption`), **bukan** langsung dari PJM; kami tidak memverifikasi ke sumber asli.
Zona waktu timestamp tidak dinyatakan di sumber; kami memperlakukannya sebagai waktu lokal.

**Pembersihan.** 145,366 baris mentah (4 timestamp ganda, 30 jam hilang, semua gap 1 jam). Stempel ganda diambil yang pertama, jam hilang diinterpolasi linear (batas gap 6 jam),
lalu dipakai hanya hari penuh: **2002-01-02 s.d. 2018-08-02 = 145,368 jam / 6,057 hari**. Rata-rata 32.079 MW, std 6.465 MW, rentang 14.544–62.009 MW.

**Skema prediksi & pencegahan data leakage.** Prediksi "diterbitkan" pukul 00:00 hari D untuk 24 jam hari D. Hanya data sampai 23:00 hari D-1 yang boleh dipakai:
semua feature beban memakai lag ≥ 24 jam atau nilai yang dibekukan pada 23:00 D-1 (`last_obs`, `d1_mean`, dst.). Tidak ada feature cuaca (tidak tersedia pada data).

**Split.** Kronologis, tanpa pengacakan: train 2002-01-16 s.d. 2014-12-31 (113,592 jam), test 2015-01-01 s.d. 2018-08-02 (31,440 jam = 21.7%).
Tuning hanya dengan `TimeSeriesSplit(4)` di dalam train. Skala/encoding dipasang hanya pada train.

**Feature** (21 pada set final): kalender (jam, hari dalam pekan, bulan, libur federal AS menurut library `holidays`, H-1/H+1 libur); riwayat (lag 24/48/72/168/336 jam, rata-rata jam yang sama 7 hari terakhir, rata-rata 24 jam);
dan *feature snapshot D-1* (beban pukul 23:00, 20:00, 17:00 hari D-1; rata-rata/maks/min 24 jam D-1; rata-rata 7 hari; selisih tren harian).


In [1]:
import json, os, warnings, time
os.makedirs("results", exist_ok=True)   # semua keluaran (JSON, gambar) disimpan di folder results/
import numpy as np, pandas as pd, holidays
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "font.size": 10})
RES = {}          # semua angka hasil dikumpulkan di sini -> results.json (dipakai laporan)
SEED = 42


In [2]:
raw = pd.read_csv("data_pjm/PJME_hourly.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
print("baris mentah:", len(raw), "|", raw.index.min(), "->", raw.index.max())
print("timestamp ganda:", raw.index.duplicated().sum())
y = raw["PJME_MW"]
y = y[~y.index.duplicated(keep="first")]                      # tanggal ganda (pergantian DST) -> ambil yang pertama
full = pd.date_range(y.index.min(), y.index.max(), freq="h")
missing = full.difference(y.index)
print("jam hilang:", len(missing), "| gap terpanjang (jam):",
      int(pd.Series(missing).diff().ne(pd.Timedelta("1h")).cumsum().value_counts().max()) if len(missing) else 0)
y = y.reindex(full).interpolate(limit=6)                      # interpolasi linear hanya untuk gap kecil
y = y["2002-01-02":"2018-08-02"]                              # hanya hari penuh (00:00-23:00)
print("sisa NaN:", int(y.isna().sum()), "| jam akhir:", len(y), "| hari:", len(y)//24)
print(y.describe().round(0))
RES["data"] = dict(rows_raw=int(len(raw)), dup=int(raw.index.duplicated().sum()), missing=int(len(missing)),
                   hours=int(len(y)), days=int(len(y)//24), start=str(y.index[0].date()), end=str(y.index[-1].date()),
                   mean=float(y.mean()), std=float(y.std()), mn=float(y.min()), mx=float(y.max()))


baris mentah: 145366 | 2002-01-01 01:00:00 -> 2018-08-03 00:00:00
timestamp ganda: 4
jam hilang: 30 | gap terpanjang (jam): 1
sisa NaN: 0 | jam akhir: 145368 | hari: 6057
count    145368.0
mean      32079.0
std        6465.0
min       14544.0
25%       27571.0
50%       31420.0
75%       35649.0
max       62009.0
Name: PJME_MW, dtype: float64


### 3.2 Pra-proses seluruh wilayah (data baru: 9 wilayah PJM lain)
Berkas tambahan berformat sama dengan PJME (kolom `Datetime` dan beban MW). Sepuluh seri dipakai: PJME, AEP, COMED, DAYTON, DEOK, DOM, DUQ, EKPC, FE, PJMW. Dua seri dikecualikan: **NI** (berhenti Januari 2011) dan **PJM_Load** (berhenti Januari 2002); cakupan wilayahnya berbeda dari seri lain dan tidak ada data baru untuk periode uji.
Aturan pra-proses yang sama untuk semua wilayah, dan setiap langkah dihitung dampaknya di tabel di bawah:
1. timestamp ganda (pergantian jam musim) → ambil yang pertama; 2. jam yang hilang → NaN; 3. nilai tak masuk akal (≤ 0, atau < 35% dari median lokal pada jam-dalam-pekan yang sama, jendela 9 minggu) → NaN; 4. gap ≤ 6 jam → interpolasi linear; 5. hanya hari penuh dipakai.
Nilai antara 35% dan 50% dari median lokal **dicatat tetapi dibiarkan**, karena tidak bisa dibuktikan salah. Nilai tinggi (> 1,6× median lokal) tidak dibuang: puncak beban nyata (cuaca ekstrem) wajar bernilai tinggi.


In [3]:
import os
DATA_DIR = "data_pjm"        # folder berisi <WILAYAH>_hourly.csv (kolom Datetime + beban MW)
REGIONS = ["PJME", "AEP", "COMED", "DAYTON", "DEOK", "DOM", "DUQ", "EKPC", "FE", "PJMW"]
EXCLUDED = {"NI": "seri berhenti Januari 2011 (cakupan wilayah berbeda dari seri lain)", "PJM_Load": "seri berhenti Januari 2002 (cakupan wilayah berbeda)"}
SPLIT_MR = "2016-01-01"      # periode uji bersama semua wilayah: 2016-01-01 s.d. 2018-08-02
LOW_FRAC, BORDER_FRAC = 0.35, 0.50
RES2 = {"split": SPLIT_MR, "excluded": EXCLUDED, "regions": {}}
os.makedirs("results/regions_out", exist_ok=True)


In [4]:
def load_raw(code):
    s = pd.read_csv(f"{DATA_DIR}/{code}_hourly.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
    return s.iloc[:, 0].astype(float)

def clean_region(code):
    """Pra-proses satu wilayah: stempel ganda -> ambil pertama; jam hilang -> NaN; nilai tak masuk akal (<=0 atau < 35% median
    jam-dalam-pekan lokal) -> NaN; gap <= 6 jam diinterpolasi linear; hanya hari penuh dipakai. Mengembalikan (seri, log)."""
    s = load_raw(code); n_raw = int(len(s)); n_dup = int(s.index.duplicated().sum()); s = s[~s.index.duplicated(keep="first")]
    full = pd.date_range(s.index.min(), s.index.max(), freq="h"); yy = s.reindex(full); n_missing = int(yy.isna().sum())
    how = yy.index.dayofweek * 24 + yy.index.hour
    med = yy.groupby(how).transform(lambda x: x.rolling(9, center=True, min_periods=3).median()); ratio = yy / med
    low = ((yy <= 0) | (ratio < LOW_FRAC)).fillna(False); border = ((ratio >= LOW_FRAC) & (ratio < BORDER_FRAC)).fillna(False)
    anomalies = [dict(t=str(i), nilai=float(yy[i]), rasio=float(ratio[i]) if np.isfinite(ratio[i]) else 0.0) for i in yy.index[low]]
    yc = yy.mask(low); na = yc.isna(); gid = (na != na.shift()).cumsum(); glen = na.groupby(gid).sum(); gfirst = na.groupby(gid).first()
    glen = glen[gfirst]; n_gaps = int(len(glen)); longest = int(glen.max()) if n_gaps else 0; n_long = int((glen > 6).sum())
    yc = yc.interpolate(limit=6)
    start = yc.index[0].ceil("D"); end = yc.index[-1] if yc.index[-1].hour == 23 else yc.index[-1].floor("D") - pd.Timedelta(hours=1)
    yc = yc[start:end]
    log = dict(rows_raw=n_raw, dup=n_dup, start=str(yc.index[0].date()), end=str(yc.index[-1].date()), hours=int(len(yc)), days=int(len(yc) // 24),
               missing_hours=n_missing, anomalies=anomalies, n_anomalies=len(anomalies), borderline=int(border.sum()), n_gaps=n_gaps, longest_gap=longest,
               gaps_over_6h=n_long, border_days={str(k): int(v) for k, v in border[border].groupby(border[border].index.date).size().sort_values(ascending=False).head(4).items()}, nan_left=int(yc.isna().sum()), mean=float(yc.mean()), mn=float(yc.min()), mx=float(yc.max()), raw_min=float(s.min()))
    return yc, log


In [5]:
# ---- Audit & pra-proses seluruh wilayah ----
CLEAN = {}
for code in REGIONS:
    CLEAN[code], lg = clean_region(code); RES2["regions"][code] = dict(audit=lg)
aud = pd.DataFrame({c: {k: v for k, v in RES2["regions"][c]["audit"].items() if k != "anomalies"} for c in REGIONS}).T
print(aud[["rows_raw", "start", "end", "dup", "missing_hours", "n_gaps", "longest_gap", "n_anomalies", "borderline", "nan_left", "days", "mean"]].round(0).to_string())
for c in REGIONS:
    for a in RES2["regions"][c]["audit"]["anomalies"][:5]: print(f"  {c}: nilai tak masuk akal dibuang -> {a['t']}  {a['nilai']:.0f} MW (rasio terhadap median lokal {a['rasio']:.2f})")


       rows_raw       start         end dup missing_hours n_gaps longest_gap n_anomalies borderline nan_left  days          mean
PJME     145366  2002-01-02  2018-08-02   4            30     30           1           0          0        0  6057  32079.065207
AEP      121273  2004-10-02  2018-08-02   4            27     27           1           0          0        0  5053  15499.387014
COMED     66497  2011-01-02  2018-08-02   4            11     11           1           0          0        0  2770  11420.266223
DAYTON   121275  2004-10-02  2018-08-02   4            25     25           1           0          1        0  5053   2037.792462
DEOK      57739  2012-01-02  2018-08-02   4             9      9           1           0         30        0  2405   3105.240696
DOM      116189  2005-05-02  2018-08-02   4            23     24           1           1          0        0  4841  10949.575987
DUQ      119068  2005-01-02  2018-08-02   4            24     24           1           0         

**Hasil audit.** Setiap wilayah punya 4 timestamp ganda (pergantian jam musim) dan hanya gap 1 jam (gap terpanjang 1 jam), jadi interpolasi dibatasi pada kasus paling ringan.
Nilai tak masuk akal yang dibuang: **DOM** 2009-12-12 00 (1.253 MW, 14% dari median lokal); **FE** 2011-06-01 01 (0 MW, 0% dari median lokal); **PJMW** 2003-05-29 00 (487 MW, 10% dari median lokal).
Nilai diragukan yang **dibiarkan**: DAYTON (1 jam; terbanyak pada 2008-09-14 (1 jam)); DEOK (30 jam; terbanyak pada 2018-04-22 (21 jam), 2018-04-24 (6 jam), 2017-11-05 (1 jam), 2018-04-23 (1 jam)); EKPC (10 jam; terbanyak pada 2013-09-29 (9 jam), 2013-09-30 (1 jam)). Yang jatuh di periode uji (mulai 2016-01-01) dapat menaikkan error wilayah itu sedikit: DEOK; sisanya jatuh di periode train. Kami tidak dapat memeriksanya ke sumber asli.


In [6]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ym = y.resample("MS").mean(); ax[0].plot(ym.index, ym.values, lw=1); ax[0].set_title("Rata-rata beban bulanan (MW)")
tmp = pd.DataFrame({"y": y}); tmp["hour"] = tmp.index.hour; tmp["dow"] = tmp.index.dayofweek; tmp["season"] = tmp.index.month.map(lambda m: "Des-Feb" if m in (12,1,2) else "Jun-Agu" if m in (6,7,8) else "Lainnya")
for s, g in tmp.groupby("season"): ax[1].plot(g.groupby("hour").y.mean(), label=s)
ax[1].set_title("Profil rata-rata per jam menurut musim"); ax[1].set_xlabel("jam"); ax[1].legend()
ax[2].bar(range(7), tmp.groupby("dow").y.mean().values); ax[2].set_xticks(range(7)); ax[2].set_xticklabels(["Sen","Sel","Rab","Kam","Jum","Sab","Min"])
ax[2].set_ylim(25000, 35000); ax[2].set_title("Rata-rata per hari dalam pekan (MW)")
plt.tight_layout(); plt.savefig("results/fig_eda.png", bbox_inches="tight"); plt.show()
RES["eda"] = dict(dow_mean=[float(v) for v in tmp.groupby("dow").y.mean().values])


**Catatan EDA.** Pola khas: beban terendah dini hari, dua musim puncak (musim panas dan dingin, kemungkinan karena pendingin dan pemanas ruang), dan akhir pekan lebih rendah
(rata-rata Sabtu 30.163 MW dan Minggu 29.406 MW, dibanding Senin 32.672 MW).


## 4. Implementation
### 4.1 Clustering profil beban harian
Satu baris = satu hari, 24 kolom = beban tiap jam. Ini analisis **deskriptif** (memakai data hari itu sendiri), bukan prediksi, sehingga tidak ada isu data leakage. Bagian ini terdiri dari: (a) tuning *preprocessing* dan perbandingan algoritma, (b) pemilihan konfigurasi akhir, (c) hasil dan interpretasinya.

#### 4.1.1 Rancangan tuning
**Yang dibandingkan** (data PJME, 265 konfigurasi, k = 2..6):
- *Preprocessing* (10): MW mentah; standardisasi per jam (konfigurasi awal); *robust scaling*; *min-max*; log lalu standardisasi; bentuk saja (z-score per hari); rasio terhadap rata-rata harian; bentuk + level; PCA 95% variansi; PCA 3 komponen.
- Algoritma (6): K-Means, *Gaussian Mixture* (kovarians *full* dan *diagonal*), *Agglomerative* (Ward dan *average*), *Spectral clustering*.
- Ukuran: Silhouette, Davies-Bouldin, Calinski-Harabasz, **stabilitas** (rata-rata ARI antara dua *subsample* 70% pada hari yang sama), dan **R² rekonstruksi** (seberapa baik rata-rata *cluster* mewakili profil MW tiap hari).

**Masalah perbandingan yang adil.** Silhouette dihitung di ruang feature masing-masing konfigurasi, sehingga tidak bisa dibandingkan antar-*preprocessing* (jarak di ruang berbeda). Karena itu dipakai dua ruang pembanding netral: S1 (standardisasi per jam) dan S2 (bentuk saja). `score` = rata-rata Silhouette di S1 dan S2.

**Aturan seleksi (ditetapkan sebelum melihat hasil):** pilih `score` tertinggi dengan stabilitas ≥ 0,8 dan *cluster* terkecil ≥ 5% hari. Sebanyak 138 dari 265 konfigurasi memenuhi syarat.


In [7]:
# Satu baris = satu hari; 24 kolom = beban tiap jam (MW). Clustering bersifat DESKRIPTIF (memakai data hari itu sendiri).
daily = y.groupby(y.index.date).apply(lambda s: s.values)
M = np.vstack(daily.values); dates = pd.to_datetime(daily.index)
print("matriks profil harian:", M.shape)


matriks profil harian: (6057, 24)


In [8]:
# ---- Fungsi bantu untuk tuning clustering: pra-pemrosesan x algoritma x k ----
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score, pairwise_distances

def cluster_preps(M):
    """Variasi pra-pemrosesan profil harian (24 nilai per hari)."""
    P = {}
    P["raw_MW"] = M.copy()                                                  # tanpa penskalaan (satuan MW)
    P["std_per_jam (awal)"] = StandardScaler().fit_transform(M)             # konfigurasi awal proyek
    P["robust_per_jam"] = RobustScaler().fit_transform(M)
    P["minmax_per_jam"] = MinMaxScaler().fit_transform(M)
    P["log_lalu_std"] = StandardScaler().fit_transform(np.log(M))
    mu = M.mean(1, keepdims=True); sd = M.std(1, keepdims=True)
    P["bentuk_z_per_hari"] = (M - mu) / sd                                  # hanya bentuk profil (level dibuang)
    P["bentuk_rasio_rata2"] = M / mu
    P["bentuk_z + level"] = np.hstack([P["bentuk_z_per_hari"], StandardScaler().fit_transform(mu) * 2.0])
    P["pca_95%_var"] = PCA(0.95, random_state=SEED).fit_transform(P["std_per_jam (awal)"])
    P["pca_3_komponen"] = PCA(3, random_state=SEED).fit_transform(P["std_per_jam (awal)"])
    return P

def cluster_make(alg, k, seed=SEED):
    if alg == "kmeans": return KMeans(k, n_init=10, random_state=seed)
    if alg == "gmm_full": return GaussianMixture(k, covariance_type="full", n_init=3, random_state=seed, reg_covar=1e-4)
    if alg == "gmm_diag": return GaussianMixture(k, covariance_type="diag", n_init=3, random_state=seed)
    if alg == "agglo_ward": return AgglomerativeClustering(k, linkage="ward")
    if alg == "agglo_average": return AgglomerativeClustering(k, linkage="average")
    if alg == "spectral": return SpectralClustering(k, affinity="nearest_neighbors", n_neighbors=15, random_state=seed, assign_labels="kmeans")
    raise ValueError(alg)

def cluster_stability(alg, k, X, n_rep=5):
    """Stabilitas = rata-rata Adjusted Rand Index antara dua model yang dilatih pada dua subsampel 70% berbeda (dibandingkan pada hari yang sama)."""
    rs = np.random.RandomState(0); n = len(X); out = []
    for r in range(n_rep):
        a = rs.choice(n, int(0.7 * n), replace=False); b = rs.choice(n, int(0.7 * n), replace=False); common = np.intersect1d(a, b)
        la = cluster_make(alg, k, r).fit_predict(X[a]); lb = cluster_make(alg, k, r + 100).fit_predict(X[b])
        ia = {j: i for i, j in enumerate(a)}; ib = {j: i for i, j in enumerate(b)}
        out.append(adjusted_rand_score([la[ia[j]] for j in common], [lb[ib[j]] for j in common]))
    return float(np.mean(out))

def cluster_recon(M, lab):
    """R2 rekonstruksi: seberapa besar variasi profil harian (dalam MW) yang dijelaskan oleh profil rata-rata tiap cluster."""
    mu = M.mean(0); sst = ((M - mu) ** 2).sum(); sse = sum(((M[lab == c] - M[lab == c].mean(0)) ** 2).sum() for c in np.unique(lab))
    return float(1 - sse / sst), float(np.sqrt(sse / M.size))

def cluster_grid(M, dts, P, plan, ks=(2, 3, 4, 5, 6), verbose=True):
    """plan = daftar (nama_prep, algoritma, daftar_k). Metrik:
       sil_S1 / sil_S2 = Silhouette pada dua ruang pembanding netral (S1 = profil distandarkan per jam, S2 = bentuk profil per hari);
       score = rata-rata sil_S1 dan sil_S2; stab = stabilitas (ARI); recon_R2 = R2 rekonstruksi profil dalam MW."""
    n = len(M); D1 = pairwise_distances(P["std_per_jam (awal)"]).astype(np.float32); D2 = pairwise_distances(P["bentuk_z_per_hari"]).astype(np.float32)
    wk = (dts.dayofweek >= 5).astype(int); mo = dts.month.values; rows = []; t0 = time.time()
    for pn, alg, kk in plan:
        X = P[pn]; Dm = pairwise_distances(X).astype(np.float32)
        for k in kk:
            lab = cluster_make(alg, k).fit_predict(X)
            if len(np.unique(lab)) < 2: continue
            s1 = silhouette_score(D1, lab, metric="precomputed"); s2 = silhouette_score(D2, lab, metric="precomputed"); r2, rm = cluster_recon(M, lab)
            rows.append(dict(prep=pn, alg=alg, k=k, sil_own=float(silhouette_score(Dm, lab, metric="precomputed")), sil_S1=float(s1), sil_S2=float(s2), score=float((s1 + s2) / 2),
                             db=float(davies_bouldin_score(X, lab)), ch=float(calinski_harabasz_score(X, lab)), stab=cluster_stability(alg, k, X), min_frac=float(np.bincount(lab).min() / n),
                             recon_R2=r2, recon_RMSE_MW=rm, nmi_month=float(normalized_mutual_info_score(mo, lab)), nmi_weekend=float(normalized_mutual_info_score(wk, lab))))
        if verbose: print(f"{pn:22s} {alg:14s} selesai ({time.time() - t0:.0f} dtk)")
    return pd.DataFrame(rows)

def cluster_final(M, ks=range(2, 7)):
    """Konfigurasi terpilih: K-Means pada profil dalam MW (tanpa standardisasi); k dengan Silhouette tertinggi.
       Konfigurasi awal (K-Means + standardisasi per jam) disimpan sebagai pembanding."""
    Xstd = StandardScaler().fit_transform(M); srows = []
    for k in ks:
        lb = KMeans(k, n_init=10, random_state=SEED).fit_predict(M); lb0 = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xstd)
        srows.append(dict(k=k, silhouette=float(silhouette_score(M, lb)), davies_bouldin=float(davies_bouldin_score(M, lb)), min_cluster=int(np.bincount(lb).min()),
                          silhouette_std=float(silhouette_score(Xstd, lb0)), ari_vs_std=float(adjusted_rand_score(lb, lb0))))
    kb = max(srows, key=lambda r: r["silhouette"])["k"]; lab = KMeans(kb, n_init=10, random_state=SEED).fit_predict(M)
    order = np.argsort([M[lab == c].mean() for c in range(kb)]); rm = {int(o): i for i, o in enumerate(order)}
    return srows, kb, np.array([rm[int(l)] for l in lab])


In [9]:
# ---- Tuning clustering (grid ringkas dijalankan di sini) ----
# Pra-pemrosesan: 10 variasi (K-Means) | Algoritma lain pada data standar: GMM diagonal, Agglomerative Ward (k = 2..6), Spectral (k = 2, 3)
CP = cluster_preps(M)
plan = [(pn, "kmeans", (2, 3, 4, 5, 6)) for pn in CP] + [("std_per_jam (awal)", "gmm_diag", (2, 3, 4, 5, 6)), ("std_per_jam (awal)", "agglo_ward", (2, 3, 4, 5, 6)), ("std_per_jam (awal)", "spectral", (2, 3))]
ct = cluster_grid(M, dates, CP, plan)
ok = ct[(ct.stab >= 0.8) & (ct.min_frac >= 0.05)]
best_k = ct.loc[ct.groupby(["prep", "alg"]).score.idxmax()].sort_values("score", ascending=False)
show = ["prep", "alg", "k", "score", "sil_own", "db", "stab", "min_frac", "recon_R2"]
print("\nKonfigurasi terbaik per (pra-pemrosesan, algoritma), diurutkan menurut skor ruang netral:")
print(best_k[show].round(3).to_string(index=False))
RES["cluster_tuning"] = dict(grid_ringkas=ct.round(4).to_dict("records"), n_eligible=int(len(ok)), n_total=int(len(ct)))


raw_MW                 kmeans         selesai (7 dtk)


std_per_jam (awal)     kmeans         selesai (15 dtk)


robust_per_jam         kmeans         selesai (23 dtk)


minmax_per_jam         kmeans         selesai (30 dtk)


log_lalu_std           kmeans         selesai (37 dtk)


...

pca_95%_var            kmeans         selesai (64 dtk)


pca_3_komponen         kmeans         selesai (69 dtk)


std_per_jam (awal)     gmm_diag       selesai (85 dtk)


std_per_jam (awal)     agglo_ward     selesai (127 dtk)


std_per_jam (awal)     spectral       selesai (153 dtk)

Konfigurasi terbaik per (pra-pemrosesan, algoritma), diurutkan menurut skor ruang netral:
              prep        alg  k  score  sil_own    db  stab  min_frac  recon_R2
std_per_jam (awal)   spectral  2  0.282    0.472 0.821 0.844     0.327     0.568
  bentuk_z + level     kmeans  2  0.279    0.335 1.183 0.979     0.332     0.552
            raw_MW     kmeans  2  0.276    0.494 0.780 0.988     0.344     0.573
std_per_jam (awal) agglo_ward  2  0.273    0.482 0.797 0.537     0.264     0.520
    minmax_per_jam     kmeans  2  0.273    0.480 0.807 0.989     0.355     0.570
    robust_per_jam     kmeans  2  0.273    0.480 0.808 0.990     0.352     0.568
std_per_jam (awal)     kmeans  2  0.271    0.477 0.809 0.972     0.363     0.562
    pca_3_komponen     kmeans  2  0.271    0.489 0.784 0.977     0.364     0.561
       pca_95%_var     kmeans  2  0.271    0.489 0.784 0.977     0.364     0.561
      log_lalu_std     kmeans  2  0.260    

Sel di atas menjalankan **grid ringkas** di dalam notebook (semua *preprocessing* dengan K-Means, plus GMM-diag, Ward, dan *spectral* pada data standar). Grid penuh (265 konfigurasi) dan pengujian pada 9 wilayah lain dijalankan lewat skrip terpisah `cluster_tune.py`, `cluster_pairs.py`, dan `cluster_regions.py` (hasilnya `cluster_tune_*.json`, dimuat oleh sel berikut) karena memakan waktu lama.


In [10]:
# ---- Hasil grid lengkap (dijalankan terpisah dengan cluster_tune.py, +-45 menit) dan perbandingan antar-wilayah ----
import glob, os
full = pd.DataFrame(json.load(open("cluster_tuning/cluster_tune_PJME.json")))                  # 265 konfigurasi: 10 pra-pemrosesan x 6 algoritma x k = 2..6
okf = full[(full.stab >= 0.8) & (full.min_frac >= 0.05)].sort_values("score", ascending=False)
print("Grid lengkap PJME:", len(full), "konfigurasi,", len(okf), "memenuhi syarat (stabilitas >= 0,8 dan cluster terkecil >= 5% hari)")
cols = ["prep", "alg", "k", "score", "sil_own", "db", "stab", "min_frac", "recon_R2"]
print("\n10 teratas menurut aturan seleksi awal:"); print(okf[cols].head(10).round(3).to_string(index=False))
print("\nKonfigurasi awal (K-Means + standardisasi per jam):"); print(full[(full.prep == "std_per_jam (awal)") & (full.alg == "kmeans")][cols].round(3).to_string(index=False))
pairs = json.load(open("cluster_tuning/cluster_pairs_PJME.json"))
print("\nPerbandingan berpasangan pada 10 subsampel 70% (k = 2), selisih skor terhadap konfigurasi awal:")
for o in pairs: print(f"  {o['prep']:22s} {o['alg']:12s} skor {o['mean']:.3f} | selisih {o['diff_vs_kmeans_std']:+.4f} +- {o['diff_sd']:.4f} | lebih baik pada {o['wins']}/10 subsampel")
# Perbandingan 5 konfigurasi kandidat pada 10 wilayah (tiap konfigurasi memakai k terbaik menurut skor, di antara yang memenuhi syarat)
CAND = [("std_per_jam (awal)", "kmeans"), ("raw_MW", "kmeans"), ("bentuk_z + level", "kmeans"), ("std_per_jam (awal)", "agglo_ward"), ("std_per_jam (awal)", "gmm_diag")]
reg_rows = []
for f in sorted(glob.glob("cluster_tuning/cluster_tune_*.json")):
    reg = os.path.basename(f).replace("cluster_tune_", "").replace(".json", "")
    for r in json.load(open(f)): r["wilayah"] = reg; reg_rows.append(r)
rg = pd.DataFrame(reg_rows); rg = rg[[(p, a) in CAND for p, a in zip(rg.prep, rg.alg)]]
best_rg = []
for (reg, pn, alg), g in rg.groupby(["wilayah", "prep", "alg"]):
    e = g[(g.stab >= 0.8) & (g.min_frac >= 0.05)]; best_rg.append((e if len(e) else g).sort_values("score", ascending=False).iloc[0])
best_rg = pd.DataFrame(best_rg)
agg = best_rg.groupby(["prep", "alg"])[["score", "sil_own", "db", "stab", "recon_R2", "min_frac"]].mean().reindex(CAND)
print("\nRata-rata pada", best_rg.wilayah.nunique(), "wilayah:"); print(agg.round(3).to_string())
k2 = rg[rg.k == 2].pivot_table(index="wilayah", columns=["prep", "alg"], values=["score", "recon_R2", "stab", "db"])
cmp_raw = {m: int((k2[m][("raw_MW", "kmeans")] > k2[m][("std_per_jam (awal)", "kmeans")]).sum()) for m in ["score", "recon_R2", "stab"]}
cmp_raw["db_lebih_kecil"] = int((k2["db"][("raw_MW", "kmeans")] < k2["db"][("std_per_jam (awal)", "kmeans")]).sum())
print("\nK-Means tanpa standardisasi vs standardisasi (k = 2, per wilayah, dari 10): ", cmp_raw)
RES["cluster_tuning"].update(
    pjme_top10=okf[cols].head(10).round(4).to_dict("records"), pjme_awal=full[(full.prep == "std_per_jam (awal)") & (full.alg == "kmeans")][cols].round(4).to_dict("records"),
    pjme_n_total=int(len(full)), pjme_n_eligible=int(len(okf)), pairs=pairs,
    best_per_alg=full[(full.stab >= 0.8) & (full.min_frac >= 0.05)].sort_values("score", ascending=False).groupby("alg").head(1)[cols].round(4).to_dict("records"),
    best_per_prep=full[(full.stab >= 0.8) & (full.min_frac >= 0.05)].sort_values("score", ascending=False).groupby("prep").head(1)[cols].round(4).to_dict("records"),
    regions=[dict(prep=p, alg=a, **{k: float(v) for k, v in agg.loc[(p, a)].items()}) for p, a in CAND], n_regions=int(best_rg.wilayah.nunique()),
    score_by_region={reg: {f"{p}|{a}": float(v) for (p, a), v in g.set_index(["prep", "alg"]).score.items()} for reg, g in best_rg.groupby("wilayah")},
    raw_vs_std=cmp_raw)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
lab_cfg = {"std_per_jam (awal)|kmeans": "K-Means\nstandar (awal)", "raw_MW|kmeans": "K-Means\nMW asli", "bentuk_z + level|kmeans": "K-Means\nshape + level", "std_per_jam (awal)|agglo_ward": "Agglomerative\nWard", "std_per_jam (awal)|gmm_diag": "GMM\ndiagonal"}
x = np.arange(len(CAND)); w = .26
for i, (m, nm) in enumerate([("score", "Silhouette (ruang netral)"), ("sil_own", "Silhouette (ruang sendiri)"), ("recon_R2", "R² rekonstruksi profil")]):
    ax[0].bar(x + (i - 1) * w, [agg.loc[c, m] for c in CAND], w, label=nm)
ax[0].set_xticks(x); ax[0].set_xticklabels([lab_cfg[f"{p}|{a}"] for p, a in CAND], fontsize=8); ax[0].set_title(f"Rata-rata {best_rg.wilayah.nunique()} wilayah (k terbaik tiap konfigurasi)"); ax[0].legend(fontsize=8)
ax[1].scatter(full.score, full.recon_R2, s=14, alpha=.35, label="semua konfigurasi PJME")
aw = full[(full.prep == "std_per_jam (awal)") & (full.alg == "kmeans")]; rw = full[(full.prep == "raw_MW") & (full.alg == "kmeans")]
ax[1].scatter(aw.score, aw.recon_R2, s=45, c="tab:orange", label="K-Means standar (awal)"); ax[1].scatter(rw.score, rw.recon_R2, s=45, c="tab:red", marker="s", label="K-Means MW asli (terpilih)")
ax[1].set_xlabel("skor Silhouette ruang netral"); ax[1].set_ylabel("R² rekonstruksi profil (MW)"); ax[1].set_title("PJME: kualitas pemisahan vs kemampuan mewakili profil"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/fig_cluster_tuning.png", bbox_inches="tight"); plt.show()


Grid lengkap PJME: 265 konfigurasi, 138 memenuhi syarat (stabilitas >= 0,8 dan cluster terkecil >= 5% hari)

10 teratas menurut aturan seleksi awal:
              prep      alg  k  score  sil_own    db  stab  min_frac  recon_R2
std_per_jam (awal) spectral  2  0.282    0.472 0.821 0.844     0.327     0.568
  bentuk_z + level   kmeans  2  0.279    0.335 1.183 0.979     0.332     0.552
            raw_MW   kmeans  2  0.276    0.494 0.780 0.988     0.344     0.573
    minmax_per_jam   kmeans  2  0.273    0.480 0.807 0.989     0.355     0.570
    robust_per_jam   kmeans  2  0.273    0.480 0.808 0.990     0.352     0.568
  bentuk_z + level gmm_diag  2  0.273    0.277 1.369 0.965     0.326     0.273
 bentuk_z_per_hari gmm_diag  2  0.271    0.307 1.064 0.943     0.336     0.254
std_per_jam (awal)   kmeans  2  0.271    0.477 0.809 0.972     0.363     0.562
    pca_3_komponen   kmeans  2  0.271    0.489 0.784 0.977     0.364     0.561
       pca_95%_var   kmeans  2  0.271    0.489 0.784 0.977   

#### 4.1.2 Hasil tuning dan pemilihan konfigurasi
- **Aturan yang dideklarasikan akan memilih** *spectral* pada *std_per_jam (awal)* (k = 2, `score` 0,282). Selisihnya terhadap K-Means standar kecil: +0,007 ± 0,003 (*paired* 70%-*subsample*). Stabilitasnya 0,84 (lebih rendah daripada K-Means, 0,97), dan *spectral clustering* tidak punya fungsi `predict` untuk hari baru.
- **Bentuk + level** (K-Means) menghasilkan `score` serupa (+0,007), dan R² rekonstruksi di PJME hampir sama (0,55 vs 0,56 pada konfigurasi awal), tetapi rata-rata 10 wilayah jauh lebih rendah (0,41 vs 0,55): *cluster*-nya memisahkan hari menurut bentuk sehingga kurang mewakili profil MW. Skor S2-nya juga cenderung menguntungkan konfigurasi ini karena ruang fiturnya mirip S2.
- **Keputusan: K-Means pada MW tanpa standardisasi.** Ini **bukan** pemenang `score` tertinggi menurut aturan di atas; ini keputusan pertimbangan yang diambil setelah melihat hasil, dengan alasan: selisih ketiganya kecil, K-Means mentah lebih stabil daripada *spectral* dan bentuk + level (0,99 vs 0,84 dan 0,98), mewakili profil MW sebaik K-Means standar atau lebih baik (R² 0,57), bisa memberi label hari baru, dan paling sederhana. Di 10 wilayah, dibanding standardisasi per jam, MW mentah lebih tinggi pada R² rekonstruksi di **10 dari 10** wilayah dan pada `score` di **7 dari 10**; selisih `score` PJME +0,005 (menang di 10 dari 10 *resample*).
- **Tidak ada model yang jelas lebih baik.** Pada PJME semua algoritma memilih k = 2, *preprocessing* hampir tidak mengubah hasil K-Means (`score` 0,271–0,276), dan proyeksi PCA tidak menunjukkan celah antar *cluster*. Perubahan dari konfigurasi awal berupa perbaikan kecil yang konsisten, bukan lompatan.
- **Silhouette bergantung pada ruang feature.** Angka Silhouette konfigurasi baru (0,494) dan awal (0,477) dihitung di ruangnya masing-masing, jadi selisihnya tidak bisa dibaca sebagai "model baru lebih bagus sebesar itu"; ukuran yang bisa dibandingkan adalah `score` dan R² rekonstruksi di atas. Kedua partisi mirip (ARI 0,82).


In [11]:
# ---- Konfigurasi terpilih: K-Means pada profil dalam MW (tanpa standardisasi); k dari Silhouette tertinggi ----
srows, K_BEST, lab = cluster_final(M)
sil = pd.DataFrame(srows); print(sil.round(3).to_string(index=False))
RES["silhouette"] = sil.round(4).to_dict("records")
names = {0: "Beban rendah", 1: "Beban tinggi"} if K_BEST == 2 else {i: f"Cluster {i}" for i in range(K_BEST)}
info = pd.DataFrame({"date": dates, "cl": lab}); info["dow"] = info.date.dt.dayofweek; info["month"] = info.date.dt.month
info["weekend"] = (info.dow >= 5).astype(int); info["mean_mw"] = M.mean(1); info["peak_h"] = M.argmax(1)
hol = holidays.US(years=range(2001, 2020)); info["holiday"] = info.date.dt.date.map(lambda x: int(x in hol))
summ = info.groupby("cl").agg(n_hari=("cl", "size"), rata2_MW=("mean_mw", "mean"), persen_akhir_pekan=("weekend", "mean"),
                              persen_libur=("holiday", "mean"), jam_puncak_median=("peak_h", "median")).round(3)
print(summ.to_string())
print("\nKomposisi bulan (% hari dalam cluster):"); print((pd.crosstab(info.cl, info.month, normalize="index") * 100).round(0).astype(int).to_string())
sk = sil.loc[sil.k == K_BEST].iloc[0]
print("Silhouette k=%d: %.3f (konfigurasi awal: %.3f) | kesesuaian dengan konfigurasi awal (ARI): %.3f" % (K_BEST, sk.silhouette, sk.silhouette_std, sk.ari_vs_std))
RES["cluster"] = dict(k=K_BEST, silhouette=float(sk.silhouette), silhouette_std=float(sk.silhouette_std), ari_vs_std=float(sk.ari_vs_std), summary=summ.reset_index().to_dict("records"),
                      month_pct=(pd.crosstab(info.cl, info.month, normalize="index") * 100).round(1).to_dict("index"))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for c in range(K_BEST):
    prof = M[lab == c]; ax[0].plot(prof.mean(0), lw=2, label=f"{names[c]} (n={len(prof)})"); ax[0].fill_between(range(24), np.percentile(prof, 10, 0), np.percentile(prof, 90, 0), alpha=.15)
ax[0].set_title("Profil beban tiap cluster (garis = rata-rata, area = P10-P90)"); ax[0].set_xlabel("jam"); ax[0].set_ylabel("MW"); ax[0].legend()
ax[1].plot(sil.k, sil.silhouette, "o-", label="K-Means MW asli (terpilih)"); ax[1].plot(sil.k, sil.silhouette_std, "s--", label="K-Means standar (awal)"); ax[1].set_title("Silhouette vs k"); ax[1].set_xlabel("k"); ax[1].legend()
pc = PCA(2).fit_transform(M); ax[2].scatter(pc[:, 0], pc[:, 1], c=lab, s=4, cmap="coolwarm"); ax[2].set_title("Proyeksi PCA 2D (warna = cluster)")
plt.tight_layout(); plt.savefig("results/fig_cluster.png", bbox_inches="tight"); plt.show()


 k  silhouette  davies_bouldin  min_cluster  silhouette_std  ari_vs_std
 2       0.494           0.780         2083           0.477       0.824
 3       0.412           0.849          904           0.383       0.781
 4       0.324           1.001          701           0.314       0.712
 5       0.344           0.949          483           0.327       0.766
 6       0.321           0.968          444           0.317       0.750
    n_hari   rata2_MW  persen_akhir_pekan  persen_libur  jam_puncak_median
cl                                                                        
0     3974  29316.132               0.348         0.035               19.0
1     2083  37350.260               0.167         0.021               17.0

Komposisi bulan (% hari dalam cluster):
month  1   2   3   4   5   6   7   8   9   10  11  12
cl                                                   
0       6   7  12  13  12   6   2   3   8  12  12   8
1      14  10   3   0   3  13  22  19   7   1   1   8
Silhouette 

**Interpretasi *cluster* (k = 2).** *Cluster* *beban tinggi* (2,083 hari, rata-rata 37.350 MW) terkonsentrasi pada Juni–Agustus dan Desember–Februari,
dengan puncak sore (median jam 17). *Cluster* *beban rendah* (3,974 hari, rata-rata 29.316 MW) berisi musim peralihan (Mar–Mei, Sep–Nov) dan lebih banyak akhir pekan
(35% vs 17%).

**Peringatan.** Silhouette 0,494 (k = 2) tertinggi, tetapi proyeksi PCA memperlihatkan **tidak ada celah** antar *cluster*: beban harian membentuk kontinum dan K-Means hanya membelahnya pada tingkat beban sekitar tengah.
Jadi *cluster* ini berguna sebagai ringkasan ("hari ekstrem musim" vs "hari moderat"), bukan bukti bahwa ada dua jenis hari yang terpisah alami. Silhouette untuk k = 3 adalah 0,412.


### 4.2 Regression: prediksi day-ahead
Pertama dibangun feature dan split, lalu baseline, lalu tuning HGB.

In [12]:
# ---- Rekayasa feature untuk prediksi DAY-AHEAD ----
# Skema: prediksi diterbitkan pukul 00:00 hari D untuk 24 jam hari D. Hanya data sampai 23:00 hari D-1 yang boleh dipakai.
# Setiap feature berbasis beban memakai lag >= 24 jam, atau nilai yang "dibekukan" pada waktu terbit (23:00 D-1).
def build_features(y):
    d = pd.DataFrame({"y": y}); ix = d.index
    d["hour"] = ix.hour; d["dow"] = ix.dayofweek; d["month"] = ix.month
    hol = holidays.US(years=range(2001, 2020)); dt = pd.Series(ix.date, index=ix)
    d["is_hol"] = dt.map(lambda x: int(x in hol)).values
    d["hol_prev"] = dt.map(lambda x: int((x - pd.Timedelta(days=1)) in hol)).values
    d["hol_next"] = dt.map(lambda x: int((x + pd.Timedelta(days=1)) in hol)).values
    for L in (24, 48, 72, 168, 336): d[f"lag{L}"] = d.y.shift(L)
    d["wk_mean"] = pd.concat([d.y.shift(24 * k) for k in range(1, 8)], axis=1).mean(axis=1)    # rata-rata jam yang sama 7 hari terakhir
    d["dmean_prev"] = d.y.shift(24).rolling(24).mean()
    # feature "saat terbit": nilai pada 23:00 D-1 (indeks t-(hour+1)) dan statistik 24 jam / 7 hari yang berakhir di titik itu
    yv = d.y.values; n = len(d); pos = np.arange(n) - (d.hour.values + 1); ok = pos >= 0; p = np.clip(pos, 0, None)
    r24 = d.y.rolling(24).mean().values; r24max = d.y.rolling(24).max().values; r24min = d.y.rolling(24).min().values; r168 = d.y.rolling(168).mean().values
    for nm, off in (("last_obs", 0), ("last_obs_m3", 3), ("last_obs_m6", 6)):
        q = pos - off; d[nm] = np.where(q >= 0, yv[np.clip(q, 0, None)], np.nan)
    d["d1_mean"] = np.where(ok, r24[p], np.nan); d["d1_max"] = np.where(ok, r24max[p], np.nan); d["d1_min"] = np.where(ok, r24min[p], np.nan)
    d["w1_mean"] = np.where(ok, r168[p], np.nan); d["d1_trend"] = d.d1_mean - np.where(ok, np.roll(r24, 24)[p], np.nan)
    return d.dropna()

d = build_features(y)
CAL = ["hour", "dow", "month", "is_hol", "hol_prev", "hol_next"]
LAG = ["lag24", "lag48", "lag168", "lag336", "dmean_prev", "wk_mean"]
ISSUE = ["last_obs", "last_obs_m3", "last_obs_m6", "d1_mean", "d1_max", "d1_min", "w1_mean", "d1_trend", "lag72"]
SETS = {"A. Kalender saja": CAL, "B. Kalender + lag >=24 jam": CAL + LAG, "C. B + fitur saat-terbit (D-1)": CAL + LAG + ISSUE}
# ---- Split kronologis ----
SPLIT = "2015-01-01"
tr, te = d[d.index < SPLIT], d[d.index >= SPLIT]
print(f"train: {tr.index.min().date()} -> {tr.index.max().date()} ({len(tr):,} jam)")
print(f"test : {te.index.min().date()} -> {te.index.max().date()} ({len(te):,} jam) | porsi test {len(te)/len(d):.1%}")
RES["split"] = dict(n_train=int(len(tr)), n_test=int(len(te)), test_share=float(len(te)/len(d)), train_start=str(tr.index.min().date()), train_end=str(tr.index.max().date()),
                    test_start=str(te.index.min().date()), test_end=str(te.index.max().date()), test_mean=float(te.y.mean()), test_std=float(te.y.std()), n_features=len(SETS["C. B + fitur saat-terbit (D-1)"]))
def metrics(t, p):
    mae = mean_absolute_error(t, p); rm = mean_squared_error(t, p) ** .5
    return dict(MAE=float(mae), RMSE=float(rm), R2=float(r2_score(t, p)), rel=float(100 * mae / np.mean(t)))


train: 2002-01-16 -> 2014-12-31 (113,592 jam)
test : 2015-01-01 -> 2018-08-02 (31,440 jam) | porsi test 21.7%


In [13]:
# ---- Uji data leakage (perturbation): rusak SELURUH nilai beban hari D dengan angka acak, hitung ulang semua feature,
# lalu cek apakah feature pada jam-jam hari D berubah. Jika ya, feature itu memakai informasi dari hari yang diprediksi (bocor).
FINAL = SETS["C. B + fitur saat-terbit (D-1)"]
rng_l = np.random.RandomState(0)
uniq_days = pd.to_datetime(d.index.date).unique()[400:-5]
test_days = rng_l.choice(uniq_days, 25, replace=False)
changed = {}
for day in test_days:
    yp = y.copy(); msk = (yp.index >= day) & (yp.index < day + pd.Timedelta(days=1))
    yp[msk] = rng_l.uniform(10000, 60000, msk.sum())
    d_pert = build_features(yp)
    rows_ = d.index[(d.index >= day) & (d.index < day + pd.Timedelta(days=1))]
    diff = (d.loc[rows_, FINAL] - d_pert.loc[rows_, FINAL]).abs().max()
    for c_ in diff[diff > 1e-9].index: changed[c_] = changed.get(c_, 0) + 1
print("fitur yang berubah saat nilai hari D dirusak:", changed if changed else "TIDAK ADA -> tidak ada kebocoran dari hari yang diramal (25 hari acak diuji)")
RES["leak_test"] = dict(n_days=25, n_features=len(FINAL), changed_features=changed)


fitur yang berubah saat nilai hari D dirusak: TIDAK ADA -> tidak ada kebocoran dari hari yang diramal (25 hari acak diuji)


**Uji data leakage (perturbation).** Seluruh nilai beban satu hari D dirusak dengan angka acak, lalu semua feature dihitung ulang. Jika ada feature pada jam-jam hari D yang ikut berubah, feature itu memakai informasi dari hari yang diprediksi. Pengujian pada 25 hari acak: lihat keluaran sel di atas (harus tidak ada feature yang berubah).

In [14]:
# ---- Baseline sederhana (tanpa belajar) ----
T = te.y.values
base = {"Persistensi harian (lag24)": te.lag24.values,
        "Persistensi mingguan (lag168)": te.lag168.values,
        "Rata-rata lag24/48/168": te[["lag24", "lag48", "lag168"]].mean(axis=1).values,
        "Rata-rata jam sama 7 hari": te.wk_mean.values}
RES["baseline"] = {k: metrics(T, v) for k, v in base.items()}
print(pd.DataFrame(RES["baseline"]).T.round(3).to_string())
BEST_BASE = min(RES["baseline"], key=lambda k: RES["baseline"][k]["MAE"]); print("\nbaseline terbaik:", BEST_BASE)


                                    MAE      RMSE     R2     rel
Persistensi harian (lag24)     2209.047  3025.216  0.780   7.053
Persistensi mingguan (lag168)  3400.302  4645.376  0.481  10.856
Rata-rata lag24/48/168         2372.253  3145.625  0.762   7.574
Rata-rata jam sama 7 hari      2710.059  3612.987  0.686   8.652

baseline terbaik: Persistensi harian (lag24)


Baseline terbaik adalah **persistensi harian** (beban jam yang sama kemarin). Model harus mengalahkannya agar dianggap punya kemampuan prediksi.

In [15]:
# ---- Tuning HistGradientBoosting: GridSearch + TimeSeriesSplit(4) HANYA pada data train ----
GRID = {"learning_rate": [0.03, 0.06], "max_iter": [400, 800], "max_leaf_nodes": [31, 63], "min_samples_leaf": [50, 100]}
FINAL = SETS["C. B + fitur saat-terbit (D-1)"]
t0 = time.time()
gs = GridSearchCV(HistGradientBoostingRegressor(random_state=SEED), GRID, cv=TimeSeriesSplit(4), scoring="neg_mean_absolute_error", n_jobs=-1).fit(tr[FINAL], tr.y)
print("parameter terbaik:", gs.best_params_, "| MAE CV (train):", round(-gs.best_score_, 1), f"| {time.time()-t0:.0f}s")
BP = gs.best_params_; RES["tuning"] = dict(best=BP, cv_mae=float(-gs.best_score_), grid_size=int(np.prod([len(v) for v in GRID.values()])))


parameter terbaik: {'learning_rate': 0.06, 'max_iter': 800, 'max_leaf_nodes': 31, 'min_samples_leaf': 50} | MAE CV (train): 1161.7 | 224s


In [16]:
# ---- Ablation study: set mana yang layak? Dinilai dengan CV KRONOLOGIS DI DALAM TRAIN (independen dari test) lalu dilaporkan juga di test ----
abl = []
for nm, cols in SETS.items():
    cvm = []
    for a, b in TimeSeriesSplit(4).split(tr):
        m = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(tr.iloc[a][cols], tr.iloc[a].y)
        cvm.append(100 * mean_absolute_error(tr.iloc[b].y, m.predict(tr.iloc[b][cols])) / tr.iloc[b].y.mean())
    m = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(tr[cols], tr.y); mt = metrics(T, m.predict(te[cols]))
    abl.append(dict(set=nm, n_fitur=len(cols), cv_rel_pct=float(np.mean(cvm)), **mt))
abl = pd.DataFrame(abl); print(abl.round(3).to_string(index=False)); RES["ablation"] = abl.to_dict("records")


                           set  n_fitur  cv_rel_pct      MAE     RMSE    R2   rel
              A. Kalender saja        6       7.727 2995.296 3766.641 0.659 9.563
    B. Kalender + lag >=24 jam       12       4.893 1743.279 2385.486 0.863 5.566
C. B + fitur saat-terbit (D-1)       21       3.589 1274.602 1868.668 0.916 4.069


**Ablation study.** Kenaikan akurasi terbesar datang dari informasi, bukan dari algoritma: kalender saja → relative error 9,6% di test; + lag ≥ 24 jam → 5,6%; + feature snapshot D-1 → 4,1%.
Urutan yang sama muncul pada CV kronologis di dalam train (7,7% → 4,9% → 3,6%), yang tidak menyentuh test.

**Catatan kejujuran.** Set feature snapshot D-1 dirancang setelah uji coba awal yang juga melihat data test, jadi angka test set C sedikit optimistis.
CV di dalam train dan rolling-origin di bawah memberi konfirmasi yang tidak bergantung pada pilihan itu.


## 5. Results
### 5.1 Perbandingan model di test (2015-2018)

In [17]:
# ---- Perbandingan model pada feature final (set C) ----
Xtr, Xte = tr[FINAL], te[FINAL]
models = {}
hgb = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(Xtr, tr.y); models["HistGradientBoosting (tuned)"] = hgb.predict(Xte)
rf = RandomForestRegressor(300, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=SEED).fit(Xtr, tr.y); models["Random Forest"] = rf.predict(Xte)
def onehot(df): return pd.get_dummies(df[FINAL].astype({"hour": "category", "dow": "category", "month": "category"}), columns=["hour", "dow", "month"], dtype=float)
Otr = onehot(tr); Ote = onehot(te).reindex(columns=Otr.columns, fill_value=0); sc = StandardScaler().fit(Otr)
rg = Ridge(alpha=10).fit(sc.transform(Otr), tr.y); models["Ridge (one-hot kalender)"] = rg.predict(sc.transform(Ote))
RES["models"] = {k: metrics(T, v) for k, v in models.items()}
tab = pd.DataFrame({**{f"[baseline] {k}": v for k, v in RES["baseline"].items()}, **RES["models"]}).T
print(tab.round(3).to_string())
best_base = RES["baseline"][BEST_BASE]; h = RES["models"]["HistGradientBoosting (tuned)"]
RES["skill"] = dict(mae_reduction_pct=float(100 * (1 - h["MAE"] / best_base["MAE"])), rmse_reduction_pct=float(100 * (1 - h["RMSE"] / best_base["RMSE"])))
print("\nPenurunan MAE vs baseline terbaik (%s): %.1f%% | RMSE: %.1f%%" % (BEST_BASE, RES["skill"]["mae_reduction_pct"], RES["skill"]["rmse_reduction_pct"]))
pred = models["HistGradientBoosting (tuned)"]


                                               MAE      RMSE     R2     rel
[baseline] Persistensi harian (lag24)     2209.047  3025.216  0.780   7.053
[baseline] Persistensi mingguan (lag168)  3400.302  4645.376  0.481  10.856
[baseline] Rata-rata lag24/48/168         2372.253  3145.625  0.762   7.574
[baseline] Rata-rata jam sama 7 hari      2710.059  3612.987  0.686   8.652
HistGradientBoosting (tuned)              1274.602  1868.668  0.916   4.069
Random Forest                             1370.797  1967.501  0.907   4.376
Ridge (one-hot kalender)                  1404.784  1942.335  0.909   4.485

Penurunan MAE vs baseline terbaik (Persistensi harian (lag24)): 42.3% | RMSE: 38.2%


In [18]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})
w0 = pd.Timestamp("2017-07-17"); sl = (te.index >= w0) & (te.index < w0 + pd.Timedelta(days=14))
ax[0].plot(te.index[sl], te.y[sl], label="Aktual", lw=1.6); ax[0].plot(te.index[sl], pred[sl], label="Prediksi HGB (day-ahead)", lw=1.2)
ax[0].plot(te.index[sl], te.lag24[sl], label="Baseline lag24", lw=.9, alpha=.6, ls="--"); ax[0].set_title("Dua pekan contoh (17-30 Jul 2017, musim panas)"); ax[0].set_ylabel("MW"); ax[0].legend(loc="upper left", fontsize=8)
ax[1].scatter(te.y, pred, s=2, alpha=.25); lo, hi = te.y.min(), te.y.max(); ax[1].plot([lo, hi], [lo, hi], "r--", lw=1); ax[1].set_xlabel("Aktual (MW)"); ax[1].set_ylabel("Prediksi (MW)"); ax[1].set_title("Aktual vs prediksi (test)")
plt.tight_layout(); plt.savefig("results/fig_forecast.png", bbox_inches="tight"); plt.show()


### 5.2 Ketahanan: rolling-origin 8 fold (uji 6 bulan per fold, train meluas)

In [19]:
# ---- Uji ketahanan: rolling-origin 8 lipatan (train meluas, uji = 6 bulan berikutnya), semua data ----
rows = []
for i, (a, b) in enumerate(TimeSeriesSplit(8, test_size=4380).split(d)):
    A_, B_ = d.iloc[a], d.iloc[b]
    m = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(A_[FINAL], A_.y); pm = m.predict(B_[FINAL]); mu = B_.y.mean()
    rows.append(dict(lipatan=i + 1, uji_mulai=str(B_.index[0].date()), uji_akhir=str(B_.index[-1].date()),
                     rel_HGB=100 * mean_absolute_error(B_.y, pm) / mu, rel_lag24=100 * mean_absolute_error(B_.y, B_.lag24) / mu, R2_HGB=r2_score(B_.y, pm)))
ro = pd.DataFrame(rows); print(ro.round(3).to_string(index=False))
print("\nrata-rata: HGB %.2f%% (std %.2f) | lag24 %.2f%% | R2 %.3f" % (ro.rel_HGB.mean(), ro.rel_HGB.std(), ro.rel_lag24.mean(), ro.R2_HGB.mean()))
RES["rolling"] = dict(rel_hgb_mean=float(ro.rel_HGB.mean()), rel_hgb_std=float(ro.rel_HGB.std()), rel_lag24_mean=float(ro.rel_lag24.mean()), r2_mean=float(ro.R2_HGB.mean()),
                      hgb_wins=int((ro.rel_HGB < ro.rel_lag24).sum()), n=int(len(ro)), rows=ro.round(3).to_dict("records"))


 lipatan  uji_mulai  uji_akhir  rel_HGB  rel_lag24  R2_HGB
       1 2014-08-04 2015-02-02    3.645      6.572   0.914
       2 2015-02-02 2015-08-03    4.222      7.615   0.912
       3 2015-08-04 2016-02-02    3.423      6.395   0.941
       4 2016-02-02 2016-08-02    3.895      6.654   0.929
       5 2016-08-03 2017-02-01    3.706      6.492   0.931
       6 2017-02-01 2017-08-02    4.332      7.503   0.909
       7 2017-08-03 2018-02-01    4.034      6.801   0.906
       8 2018-02-01 2018-08-02    4.763      7.990   0.876

rata-rata: HGB 4.00% (std 0.43) | lag24 7.00% | R2 0.915


### 5.3 Analisis error dan feature penting

In [20]:
# ---- Analisis error (set test) ----
E = te[["y"]].copy(); E["pred"] = pred; E["err"] = E.pred - E.y; E["ae"] = E.err.abs(); E["pe"] = 100 * E.ae / E.y
E["hour"] = te.hour; E["dow"] = te.dow; E["month"] = te.month; E["is_hol"] = te.is_hol; E["date"] = te.index.date
by_hour = E.groupby("hour").pe.mean(); by_dow = E.groupby("dow").pe.mean(); by_month = E.groupby("month").pe.mean()
daytype = E.assign(tipe=np.where(E.is_hol == 1, "Libur federal", np.where(E.dow >= 5, "Akhir pekan", "Hari kerja"))).groupby("tipe").pe.mean()
daily_err = E.groupby("date").agg(mape=("pe", "mean"), mean_y=("y", "mean")); worst = daily_err.sort_values("mape", ascending=False).head(10)
print("MAPE per tipe hari:", daytype.round(2).to_dict()); print("MAPE per bulan:", by_month.round(2).to_dict())
print("10 hari terburuk:\n", worst.round(2).to_string()); print("bias rata-rata (pred - aktual): %.0f MW" % E.err.mean())
wk10 = int(sum(pd.Timestamp(i).dayofweek >= 5 for i in worst.index)); print("dari 10 hari terburuk, %d jatuh pada akhir pekan" % wk10)
print("P90 galat absolut: %.0f MW | P99: %.0f MW | maks: %.0f MW" % (E.ae.quantile(.9), E.ae.quantile(.99), E.ae.max()))
RES["errors"] = dict(by_hour=[float(v) for v in by_hour.values], by_dow=[float(v) for v in by_dow.values], by_month=[float(v) for v in by_month.values],
                     daytype={k: float(v) for k, v in daytype.items()}, bias=float(E.err.mean()), p90=float(E.ae.quantile(.9)), p99=float(E.ae.quantile(.99)), max=float(E.ae.max()),
                     worst=[dict(date=str(i), mape=float(r.mape)) for i, r in worst.head(5).iterrows()], worst10_weekend=wk10)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].bar(by_hour.index, by_hour.values); ax[0].set_title("MAPE per jam (%)"); ax[0].set_xlabel("jam")
ax[1].bar(by_month.index, by_month.values); ax[1].set_title("MAPE per bulan (%)"); ax[1].set_xlabel("bulan")
ax[2].hist(E.err, bins=80); ax[2].set_title("Distribusi error (MW), prediksi - aktual"); ax[2].axvline(0, c="k", lw=.8)
plt.tight_layout(); plt.savefig("results/fig_errors.png", bbox_inches="tight"); plt.show()


MAPE per tipe hari: {'Akhir pekan': 4.3, 'Hari kerja': 3.68, 'Libur federal': 4.31}
MAPE per bulan: {1: 4.0, 2: 4.2, 3: 3.81, 4: 3.39, 5: 4.18, 6: 4.77, 7: 4.37, 8: 3.81, 9: 4.22, 10: 2.9, 11: 3.05, 12: 3.3}
10 hari terburuk:
              mape    mean_y
date                       
2017-05-20  26.40  26228.58
2018-06-03  17.88  26070.29
2016-09-24  17.45  27467.00
2015-05-20  17.24  28303.79
2018-05-05  16.76  26163.12
2018-05-27  15.25  28984.21
2015-02-15  14.22  40795.88
2017-07-29  13.86  29184.42
2015-05-13  13.27  27868.71
2016-06-12  12.81  33438.42
bias rata-rata (pred - aktual): -92 MW
dari 10 hari terburuk, 8 jatuh pada akhir pekan
P90 galat absolut: 2900 MW | P99: 6681 MW | maks: 13556 MW


In [21]:
# ---- Permutation importance (sub-sampel 6.000 jam test, MAE) ----
rng = np.random.RandomState(SEED); sub = rng.choice(len(te), 6000, replace=False)
pi = permutation_importance(hgb, te[FINAL].iloc[sub], te.y.iloc[sub], n_repeats=5, random_state=SEED, scoring="neg_mean_absolute_error")
imp = pd.Series(pi.importances_mean, index=FINAL).sort_values(ascending=False); print(imp.round(1).head(12).to_string())
RES["importance"] = {k: float(v) for k, v in imp.head(10).items()}
fig, ax = plt.subplots(figsize=(7, 4)); imp.head(12)[::-1].plot.barh(ax=ax); ax.set_title("Permutation importance (kenaikan MAE, MW)"); plt.tight_layout(); plt.savefig("results/fig_importance.png", bbox_inches="tight"); plt.show()


last_obs       3359.4
lag24          2621.7
dow             428.7
d1_mean         410.9
hour            371.2
wk_mean         348.2
w1_mean         313.1
d1_min          266.0
dmean_prev      149.6
last_obs_m6     135.6
lag168          106.4
month           105.1


### 5.4 Perluasan: pipeline yang sama untuk 10 wilayah
Setiap wilayah memperoleh model sendiri dengan pipeline yang sama (feature bebas data leakage, baseline, tuning hanya di train, rolling-origin, clustering, uji data leakage). Perbedaan dengan analisis utama PJME di atas:
- **Periode uji bersama** 2016-01-01 s.d. 2018-08-02 untuk semua wilayah (agar adil dan agar wilayah dengan data pendek seperti EKPC, mulai 2013, masih punya train ≥ 2,5 tahun). PJME diulang pada protokol ini **hanya untuk tabel perbandingan**; hasil utama PJME (uji 2015+) tetap yang di atas, sehingga angka PJME di tabel sedikit berbeda.
- **Set feature dan rentang hyperparameter tidak disetel ulang per wilayah**: dirancang pada PJME. Untuk wilayah lain ini berarti pemilihan feature tidak melihat data uji mereka; hanya hyperparameter yang dicari ulang di train (grid 8 kombinasi, `TimeSeriesSplit(4)`).
- Untuk menghemat waktu, Random Forest tidak dijalankan per wilayah (pembandingnya Ridge + 4 baseline), ablation study hanya 3 set di holdout (tanpa CV), uji data leakage pada 10 hari acak, importance pada 4.000 jam.
- Prediksi uji dan model 21 feature tiap wilayah disimpan ke `results/regions_out/` (berkas besar, tidak ikut di repo; dibutuhkan hanya untuk dashboard web).


In [22]:
def cluster_region(y):
    cnt = y.groupby(y.index.normalize()).count(); good = cnt[cnt == 24].index; Mx = y[y.index.normalize().isin(good)].values.reshape(-1, 24)
    srows, kb, lab_ = cluster_final(Mx)
    return srows, kb, lab_, Mx, pd.to_datetime(good)

def run_region(code, y, split=SPLIT_MR, n_leak=10, export=True):
    t0 = time.time(); R_ = {}; FIN = SETS["C. B + fitur saat-terbit (D-1)"]
    d_ = build_features(y); tr_, te_ = d_[d_.index < split], d_[d_.index >= split]; T_ = te_.y.values
    R_["split"] = dict(n_train=int(len(tr_)), n_test=int(len(te_)), train_start=str(tr_.index.min().date()), train_end=str(tr_.index.max().date()),
                       test_start=str(te_.index.min().date()), test_end=str(te_.index.max().date()), test_mean=float(te_.y.mean()), test_std=float(te_.y.std()))
    # baseline
    bs = {"Persistensi harian (lag24)": te_.lag24.values, "Persistensi mingguan (lag168)": te_.lag168.values,
          "Rata-rata lag24/48/168": te_[["lag24", "lag48", "lag168"]].mean(axis=1).values, "Rata-rata jam sama 7 hari": te_.wk_mean.values}
    R_["baseline"] = {k: metrics(T_, v) for k, v in bs.items()}; bb = min(R_["baseline"], key=lambda k: R_["baseline"][k]["MAE"]); R_["best_baseline"] = bb
    # tuning: hanya di train, TimeSeriesSplit(4)
    GRID_ = {"learning_rate": [0.06], "max_iter": [400, 800], "max_leaf_nodes": [31, 63], "min_samples_leaf": [50, 100]}
    gs_ = GridSearchCV(HistGradientBoostingRegressor(random_state=SEED), GRID_, cv=TimeSeriesSplit(4), scoring="neg_mean_absolute_error", n_jobs=-1).fit(tr_[FIN], tr_.y)
    bp = gs_.best_params_; R_["tuning"] = dict(best=bp, cv_mae=float(-gs_.best_score_), grid_size=8)
    # ablation study (uji holdout, 3 set)
    ab = []
    for nm, cols in SETS.items():
        m_ = HistGradientBoostingRegressor(random_state=SEED, **bp).fit(tr_[cols], tr_.y); ab.append(dict(set=nm, n_fitur=len(cols), **metrics(T_, m_.predict(te_[cols]))))
    R_["ablation"] = ab
    # model
    hgb_ = HistGradientBoostingRegressor(random_state=SEED, **bp).fit(tr_[FIN], tr_.y); ph = hgb_.predict(te_[FIN])
    oh = lambda df: pd.get_dummies(df[FIN].astype({"hour": "category", "dow": "category", "month": "category"}), columns=["hour", "dow", "month"], dtype=float)
    Otr = oh(tr_); Ote = oh(te_).reindex(columns=Otr.columns, fill_value=0); sc_ = StandardScaler().fit(Otr); rg_ = Ridge(alpha=10).fit(sc_.transform(Otr), tr_.y)
    R_["models"] = {"HistGradientBoosting (tuned)": metrics(T_, ph), "Ridge (one-hot kalender)": metrics(T_, rg_.predict(sc_.transform(Ote)))}
    h_ = R_["models"]["HistGradientBoosting (tuned)"]; b_ = R_["baseline"][bb]
    R_["skill"] = dict(mae_reduction_pct=float(100 * (1 - h_["MAE"] / b_["MAE"])), rmse_reduction_pct=float(100 * (1 - h_["RMSE"] / b_["RMSE"])))
    # rolling-origin
    nsp = int(min(8, (len(d_) - 8760) // 4380)); rows_ = []
    for i, (a, b) in enumerate(TimeSeriesSplit(nsp, test_size=4380).split(d_)):
        A_, B_ = d_.iloc[a], d_.iloc[b]; m_ = HistGradientBoostingRegressor(random_state=SEED, **bp).fit(A_[FIN], A_.y); pm_ = m_.predict(B_[FIN]); mu_ = B_.y.mean()
        rows_.append(dict(lipatan=i + 1, uji_mulai=str(B_.index[0].date()), uji_akhir=str(B_.index[-1].date()), rel_HGB=100 * mean_absolute_error(B_.y, pm_) / mu_,
                          rel_lag24=100 * mean_absolute_error(B_.y, B_.lag24) / mu_, R2_HGB=r2_score(B_.y, pm_)))
    ro_ = pd.DataFrame(rows_)
    R_["rolling"] = dict(rel_hgb_mean=float(ro_.rel_HGB.mean()), rel_hgb_std=float(ro_.rel_HGB.std()), rel_lag24_mean=float(ro_.rel_lag24.mean()), r2_mean=float(ro_.R2_HGB.mean()),
                         hgb_wins=int((ro_.rel_HGB < ro_.rel_lag24).sum()), n=int(len(ro_)), rows=ro_.round(3).to_dict("records"))
    # error
    E_ = te_[["y"]].copy(); E_["pred"] = ph; E_["err"] = E_.pred - E_.y; E_["ae"] = E_.err.abs(); E_["pe"] = 100 * E_.ae / E_.y; E_["dow"] = te_.dow; E_["is_hol"] = te_.is_hol; E_["month"] = te_.month; E_["hour"] = te_.hour; E_["date"] = te_.index.date
    dt_ = E_.assign(tipe=np.where(E_.is_hol == 1, "Libur federal", np.where(E_.dow >= 5, "Akhir pekan", "Hari kerja"))).groupby("tipe").pe.mean(); dd_ = E_.groupby("date").agg(mape=("pe", "mean")).sort_values("mape", ascending=False)
    R_["errors"] = dict(by_hour=[float(v) for v in E_.groupby("hour").pe.mean().values], by_month=[float(E_[E_.month == m].pe.mean()) if (E_.month == m).any() else 0.0 for m in range(1, 13)],
                        daytype={k: float(v) for k, v in dt_.items()}, bias=float(E_.err.mean()), p90=float(E_.ae.quantile(.9)), p99=float(E_.ae.quantile(.99)), max=float(E_.ae.max()),
                        worst=[dict(date=str(i), mape=float(r.mape)) for i, r in dd_.head(5).iterrows()], worst10_weekend=int(sum(pd.Timestamp(i).dayofweek >= 5 for i in dd_.head(10).index)))
    # importance
    rs_ = np.random.RandomState(SEED); sub_ = rs_.choice(len(te_), min(4000, len(te_)), replace=False)
    pi_ = permutation_importance(hgb_, te_[FIN].iloc[sub_], te_.y.iloc[sub_], n_repeats=3, random_state=SEED, scoring="neg_mean_absolute_error")
    imp_ = pd.Series(pi_.importances_mean, index=FIN).sort_values(ascending=False); R_["importance"] = {k: float(v) for k, v in imp_.head(10).items()}
    # clustering (hari lengkap saja): K-Means pada profil MW (konfigurasi hasil tuning), lihat cells_cluster.py
    srows, kb, lab_, Mx, dts = cluster_region(y)
    inf_ = pd.DataFrame({"date": dts, "cl": lab_}); inf_["weekend"] = (inf_.date.dt.dayofweek >= 5).astype(int); inf_["mean_mw"] = Mx.mean(1); inf_["peak_h"] = Mx.argmax(1); inf_["month"] = inf_.date.dt.month
    sm_ = inf_.groupby("cl").agg(n_hari=("cl", "size"), rata2_MW=("mean_mw", "mean"), persen_akhir_pekan=("weekend", "mean"), jam_puncak_median=("peak_h", "median")).round(3)
    sk_ = [r for r in srows if r["k"] == kb][0]
    R_["silhouette"] = srows; R_["cluster"] = dict(k=int(kb), silhouette=sk_["silhouette"], silhouette_std=sk_["silhouette_std"], ari_vs_std=sk_["ari_vs_std"], summary=sm_.reset_index().to_dict("records"),
                                                    month_pct=(pd.crosstab(inf_.cl, inf_.month, normalize="index") * 100).round(1).to_dict("index"))
    # uji data leakage (perturbation)
    rl_ = np.random.RandomState(0); ud_ = pd.to_datetime(d_.index.date).unique()[400:-5]; chg = {}
    for day in rl_.choice(ud_, n_leak, replace=False):
        yp_ = y.copy(); mk_ = (yp_.index >= day) & (yp_.index < day + pd.Timedelta(days=1)); yp_[mk_] = rl_.uniform(0.3 * y.mean(), 1.9 * y.mean(), mk_.sum())
        dp_ = build_features(yp_); rw_ = d_.index[(d_.index >= day) & (d_.index < day + pd.Timedelta(days=1))]
        df_ = (d_.loc[rw_, FIN] - dp_.reindex(rw_)[FIN]).abs().max()
        for c_ in df_[df_ > 1e-9].index: chg[c_] = chg.get(c_, 0) + 1
    R_["leak_test"] = dict(n_days=n_leak, n_features=len(FIN), changed_features=chg)
    if export:   # data web + model + fixture
        td_ = pd.DataFrame({"y": te_.y.values, "p": ph, "b": te_.lag24.values}, index=te_.index); days_ = {}
        for dd, g_ in td_.groupby(td_.index.date):
            if len(g_) == 24: days_[str(dd)] = [[int(round(v)) for v in g_[c].values] for c in ("y", "p", "b")]
        cl_ = [dict(name=f"Cluster {c}", n=int((lab_ == c).sum()), mean=[int(round(v)) for v in Mx[lab_ == c].mean(0)], p10=[int(round(v)) for v in np.percentile(Mx[lab_ == c], 10, 0)],
                    p90=[int(round(v)) for v in np.percentile(Mx[lab_ == c], 90, 0)]) for c in range(kb)]
        R_["_web"] = dict(days=days_, clusters=cl_, importance={k: round(float(v), 1) for k, v in imp_.head(10).items()})
        trees = []
        for it in hgb_._predictors:
            nd = it[0].nodes
            trees.append(dict(f=[int(x) if not lf else -1 for x, lf in zip(nd["feature_idx"], nd["is_leaf"])], t=[float(x) for x in nd["num_threshold"]], l=[int(x) for x in nd["left"]],
                              r=[int(x) for x in nd["right"]], v=[float(f"{x:.12g}") for x in nd["value"]], mg=[int(x) for x in nd["missing_go_to_left"]]))
        R_["_model"] = dict(features=FIN, base=float(hgb_._baseline_prediction.ravel()[0]), trees=trees)
        ix_ = np.random.RandomState(1).choice(len(te_), 40, replace=False)
        R_["_fixture"] = dict(X=te_[FIN].iloc[ix_].values.tolist(), y=[float(v) for v in hgb_.predict(te_[FIN].iloc[ix_])])
    R_["seconds"] = round(time.time() - t0)
    return R_


In [23]:
# ---- Jalankan pipeline yang sama untuk tiap wilayah (tuning hanya di train; uji bersama 2016-01-01 s.d. 2018-08-02) ----
for code in REGIONS:
    r_ = run_region(code, CLEAN[code], export=(code != "PJME"))        # PJME di sini hanya untuk membandingkan protokol sama; model utamanya di bagian 1-8
    RES2["regions"][code].update({k: v for k, v in r_.items() if not k.startswith("_")})
    if code != "PJME":
        json.dump(dict(web=r_["_web"], model=r_["_model"], fixture=r_["_fixture"]), open(f"results/regions_out/{code}.json", "w"), separators=(",", ":"))
    h = r_["models"]["HistGradientBoosting (tuned)"]
    print(f"{code:7s} test {r_['split']['n_test']//24:4d} hari | MAE {h['MAE']:8.1f} MW | rel {h['rel']:.2f}% | R2 {h['R2']:.3f} | vs baseline {r_['skill']['mae_reduction_pct']:.1f}% | "
          f"rolling menang {r_['rolling']['hgb_wins']}/{r_['rolling']['n']} | k={r_['cluster']['k']} | leak:{'bocor' if r_['leak_test']['changed_features'] else 'tidak ada'} | {r_['seconds']}s")


PJME    test  945 hari | MAE   1298.1 MW | rel 4.16% | R2 0.909 | vs baseline 41.4% | rolling menang 8/8 | k=2 | leak:tidak ada | 257s


AEP     test  945 hari | MAE    500.2 MW | rel 3.38% | R2 0.922 | vs baseline 46.1% | rolling menang 8/8 | k=2 | leak:tidak ada | 213s


COMED   test  945 hari | MAE    478.0 MW | rel 4.23% | R2 0.887 | vs baseline 42.1% | rolling menang 8/8 | k=2 | leak:tidak ada | 123s


DAYTON  test  945 hari | MAE     81.5 MW | rel 4.04% | R2 0.906 | vs baseline 49.9% | rolling menang 8/8 | k=2 | leak:tidak ada | 213s


DEOK    test  945 hari | MAE    145.9 MW | rel 4.68% | R2 0.880 | vs baseline 35.7% | rolling menang 8/8 | k=2 | leak:tidak ada | 133s


DOM     test  945 hari | MAE    567.9 MW | rel 5.05% | R2 0.892 | vs baseline 35.7% | rolling menang 8/8 | k=3 | leak:tidak ada | 178s


DUQ     test  945 hari | MAE     62.6 MW | rel 3.97% | R2 0.904 | vs baseline 37.7% | rolling menang 8/8 | k=2 | leak:tidak ada | 179s


EKPC    test  945 hari | MAE    104.1 MW | rel 7.05% | R2 0.843 | vs baseline 25.9% | rolling menang 8/8 | k=2 | leak:tidak ada | 70s


FE      test  945 hari | MAE    281.6 MW | rel 3.64% | R2 0.904 | vs baseline 46.8% | rolling menang 8/8 | k=2 | leak:tidak ada | 112s


PJMW    test  945 hari | MAE    221.4 MW | rel 3.95% | R2 0.902 | vs baseline 41.4% | rolling menang 8/8 | k=2 | leak:tidak ada | 247s


In [24]:
# ---- Perbandingan antarwilayah (protokol sama) ----
rows = []
for c in REGIONS:
    r = RES2["regions"][c]; h = r["models"]["HistGradientBoosting (tuned)"]; bb = r["baseline"][r["best_baseline"]]
    rows.append(dict(wilayah=c, rata2_MW=r["split"]["test_mean"], hari_uji=r["split"]["n_test"] // 24, MAE=h["MAE"], rel_HGB=h["rel"], rel_base=bb["rel"], R2=h["R2"], R2_base=bb["R2"],
                     turun_MAE_pct=r["skill"]["mae_reduction_pct"], rolling_menang=f"{r['rolling']['hgb_wins']}/{r['rolling']['n']}", rolling_rel=r["rolling"]["rel_hgb_mean"], k=r["cluster"]["k"], silhouette=r["cluster"]["silhouette"]))
cmp_ = pd.DataFrame(rows); print(cmp_.round(3).to_string(index=False)); RES2["compare"] = cmp_.round(4).to_dict("records")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
o_ = cmp_.sort_values("rata2_MW", ascending=False); xx = np.arange(len(o_))
ax[0].bar(xx - .2, o_.rel_base, .4, label="baseline terbaik"); ax[0].bar(xx + .2, o_.rel_HGB, .4, label="HGB"); ax[0].set_xticks(xx); ax[0].set_xticklabels(o_.wilayah, rotation=30); ax[0].set_ylabel("relative error (%)"); ax[0].set_title("Relative error per wilayah (urut: beban rata-rata besar -> kecil)"); ax[0].legend()
ax[1].scatter(cmp_.rata2_MW, cmp_.rel_HGB); [ax[1].annotate(w, (m, e), fontsize=9, xytext=(3, 3), textcoords="offset points") for w, m, e in zip(cmp_.wilayah, cmp_.rata2_MW, cmp_.rel_HGB)]
ax[1].set_xscale("log"); ax[1].set_xlabel("beban rata-rata test (MW, skala log)"); ax[1].set_ylabel("relative error HGB (%)"); ax[1].set_title("Relative error vs ukuran wilayah")
plt.tight_layout(); plt.savefig("results/fig_regions.png", bbox_inches="tight"); plt.show()
cor_ = float(np.corrcoef(np.log(cmp_.rata2_MW), cmp_.rel_HGB)[0, 1]); print("korelasi log(beban rata-rata) vs galat relatif HGB: %.2f (n=%d wilayah; indikatif saja)" % (cor_, len(cmp_))); RES2["size_corr"] = cor_


wilayah  rata2_MW  hari_uji      MAE  rel_HGB  rel_base    R2  R2_base  turun_MAE_pct rolling_menang  rolling_rel  k  silhouette
   PJME 31173.248       945 1298.061    4.164     7.110 0.909    0.774         41.437            8/8        3.998  2       0.494
    AEP 14782.931       945  500.211    3.384     6.281 0.922    0.753         46.124            8/8        3.284  2       0.441
  COMED 11293.229       945  477.955    4.232     7.306 0.887    0.724         42.069            8/8        3.998  2       0.533
 DAYTON  2016.057       945   81.524    4.044     8.073 0.906    0.667         49.913            8/8        3.955  2       0.450
   DEOK  3117.016       945  145.906    4.681     7.284 0.880    0.747         35.740            8/8        4.500  2       0.486
    DOM 11238.419       945  567.949    5.054     7.855 0.892    0.764         35.660            8/8        4.888  3       0.486
    DUQ  1576.974       945   62.587    3.969     6.375 0.904    0.783         37.749            

korelasi log(beban rata-rata) vs galat relatif HGB: -0.42 (n=10 wilayah; indikatif saja)


In [25]:
# ---- Ringkasan clustering antarwilayah (konfigurasi terpilih) ----
rows_ = [dict(wilayah=c, k=RES2["regions"][c]["cluster"]["k"], silhouette=RES2["regions"][c]["cluster"]["silhouette"], silhouette_std=RES2["regions"][c]["cluster"]["silhouette_std"],
              ari_vs_std=RES2["regions"][c]["cluster"]["ari_vs_std"]) for c in REGIONS]
cl_tab = pd.DataFrame(rows_); print(cl_tab.round(3).to_string(index=False)); RES2["cluster_summary"] = cl_tab.round(4).to_dict("records")


wilayah  k  silhouette  silhouette_std  ari_vs_std
   PJME  2       0.494           0.477       0.824
    AEP  2       0.441           0.447       0.947
  COMED  2       0.533           0.485       0.716
 DAYTON  2       0.450           0.446       0.868
   DEOK  2       0.486           0.469       0.790
    DOM  3       0.486           0.479       0.939
    DUQ  2       0.493           0.467       0.761
   EKPC  2       0.526           0.478       0.752
     FE  2       0.467           0.452       0.812
   PJMW  2       0.448           0.454       0.962


In [26]:
json.dump(RES2, open("results/results_regions.json", "w"), indent=1, default=float); print("results/results_regions.json tersimpan")


results/results_regions.json tersimpan


**Bacaan.** HGB mengalahkan baseline terbaik (MAE) di **10 dari 10** wilayah, dengan penurunan MAE 25,9%–49,9%; pada rolling-origin HGB menang di 80 dari 80 fold gabungan seluruh wilayah.
Relative error HGB terendah di **AEP** (3,38%) dan tertinggi di **EKPC** (7,05%). Korelasi log(beban rata-rata) dengan relative error r = -0,42: relative error tidak menunjukkan hubungan yang jelas dengan ukuran wilayah (hanya 10 titik, indikatif saja).
Uji data leakage: tidak ada feature yang berubah pada seluruh wilayah.
**Batas klaim.** Tiap wilayah diuji pada satu periode uji ≈ 2,6 tahun tanpa data cuaca; angka antarwilayah tidak diuji signifikansinya, dan kesamaan pola antarwilayah yang bertetangga (mis. cuaca yang sama) berarti hasilnya tidak independen.


## 6. Evaluation
| Ukuran | Baseline terbaik (lag24) | HistGradientBoosting |
|---|---|---|
| MAE (MW) | 2.209 | **1.275** |
| RMSE (MW) | 3.025 | **1.869** |
| R² | 0,780 | **0,916** |
| Relative error (MAE / rata-rata beban test 31.323 MW) | 7,05% | **4,07%** |

- **Cara membaca R²:** R² 0,916 menggelembung oleh siklus harian yang mudah ditebak: baseline lag24 saja sudah R² 0,780. Ukuran yang lebih jujur adalah penurunan error terhadap baseline dan relative error, bukan R² mentah.
- **Kemampuan prediksi nyata:** HGB menurunkan MAE sebesar **42,3%** dan RMSE **38,2%** dibanding persistensi harian.
- **Konsisten antarwaktu:** pada rolling-origin, HGB mengalahkan lag24 di **8 dari 8** fold; relative error rata-rata 4,00% (std 0,43) vs 7,00% untuk lag24; R² rata-rata 0,915.
  Fold terakhir (2018) paling sulit (relative error 4,76%), jadi ada indikasi error naik pada periode terbaru; penyebabnya tidak diselidiki.
- **Algoritma bukan faktor utama:** Ridge sederhana mencapai R² 0,909 (relative error 4,48%) dan Random Forest 0,907 (4,38%), selisih kecil dari HGB.
- **Error:** bias rata-rata -92 MW (cenderung memprediksi sedikit rendah), P90 error absolut 2.900 MW, P99 6.681 MW, maksimum 13.556 MW. Hari kerja 3,68%, akhir pekan 4,30%, libur federal 4,31% (MAPE).
  Hari terburuk didominasi akhir pekan (8 dari 10 hari terburuk; periksa keluaran sel di atas).
- **Clustering:** Silhouette 0,494 untuk k = 2 (K-Means pada MW tanpa standardisasi, hasil tuning); ini merupakan pembelahan kontinum (lihat Bagian 4.1). Tuning 265 konfigurasi tidak menemukan model yang jelas lebih baik.


## 7. Engineering Interpretation
1. **Mengapa akurasinya tinggi?** Beban agregat banyak pelanggan meredam perilaku acak tiap rumah, dan data per jam memberi riwayat yang sangat berkorelasi (beban sekarang sangat mirip beban jam sebelumnya).
   R² ≈ 0,92 dan relative error ≈ 4,1% di sini terutama karena sifat data, bukan karena algoritma yang lebih baik (Ridge dan Random Forest hampir sama baiknya).
2. **Informasi terbaik adalah kondisi terbaru.** Feature terpenting adalah beban pukul 23:00 hari sebelumnya dan beban jam yang sama kemarin (lihat permutation importance); hari dalam pekan, jam, dan rata-rata beban harian menyusul, sedangkan bulan relatif kecil. Ini sejalan dengan fakta bahwa beban dipengaruhi cuaca yang persisten dari hari ke hari.
3. **Keterbatasan fisik: cuaca tidak ada.** Beban sensitif terhadap suhu (pendingin/pemanas), sedangkan data ini tidak memuat prakiraan suhu. Error terbesar terjadi pada hari yang berbeda dari hari sebelumnya (terutama akhir pekan setelah hari kerja dengan cuaca berbeda).
   Dugaan ini konsisten dengan temuan, tetapi **tidak diuji** karena tidak ada data cuaca. Menambahkan prakiraan suhu adalah langkah berikutnya yang paling masuk akal.
4. **Makna error dalam MW.** MAE sekitar 1,3 GW; P99 sekitar 6,7 GW. Error sebesar ini yang harus diantisipasi operator dengan cadangan; kami tidak menghitung biaya karena tidak ada data tarif atau cadangan.
5. **Apakah hasil PJME khas?** Pada 9 wilayah lain dengan pipeline yang sama, HGB mengalahkan baseline terbaik di 10 dari 10 wilayah (relative error 3,38%–7,05%), jadi pola "beban agregat per jam sangat dapat diprediksi dari kalender dan riwayat" tidak hanya berlaku di PJME. Relative error tidak menunjukkan hubungan yang jelas dengan ukuran wilayah.
6. **Keterbatasan generalisasi.** Data satu wilayah di Amerika Serikat (iklim, kalender libur, dan perilaku konsumsi berbeda dari Indonesia). Pola yang sama tidak otomatis berlaku di Indonesia (mis. tanpa musim dingin, ada Lebaran); hal itu perlu data lokal.


## 8. Conclusion
- Dengan feature kalender + riwayat beban yang aman dari data leakage, HGB memprediksi beban per jam sehari ke depan dengan **MAE 1.275 MW (relative error 4,07%), RMSE 1.869 MW, R² 0,916** di test 2015–2018,
  **42,3% lebih baik** daripada persistensi harian, dan konsisten pada 8 dari 8 fold rolling-origin.
- Angka paling jujur adalah rata-rata rolling-origin: relative error 4,00% dan R² 0,915; angka test sedikit optimistis karena set feature snapshot D-1 dirancang setelah melihat data test.
- Akurasi yang tinggi berasal dari **jenis data** (agregat, per jam) dan **feature riwayat**, bukan dari model yang lebih kompleks; Ridge hampir sama baiknya.
- Clustering memberi ringkasan "hari ekstrem musim vs moderat" (Silhouette 0,494), tetapi datanya kontinum sehingga *cluster* bukan kelompok alami. Tuning *preprocessing* dan perbandingan 6 algoritma hanya menghasilkan perbaikan kecil; K-Means tanpa standardisasi dipilih sebagai keputusan pertimbangan (bukan pemenang `score` menurut aturan yang dideklarasikan).
- Perluasan 10 wilayah: HGB mengalahkan baseline terbaik di 10 dari 10 wilayah (penurunan MAE 25,9%–49,9%, relative error 3,38%–7,05%) tanpa menyetel ulang feature per wilayah; pra-proses hanya membuang 3 nilai tak masuk akal.
- Langkah lanjut: tambahkan prakiraan suhu, uji prediksi probabilistik (interval), dan validasi pada data beban Indonesia bila tersedia.

## 9. Daftar Pustaka
Dataset acuan (rujukan tim; berkas yang dipakai berasal dari salinan GitHub dan paket data yang diunggah, bukan langsung dari PJM):
- [D1] Kaggle: robikscube, Hourly Energy Consumption, https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption
- [D2] Kaggle: ashishkumarrana, PJM Hourly Energy Consumption, https://www.kaggle.com/datasets/ashishkumarrana/pjm-hourly-energy-consumption
- [D3] IEEE DataPort: PJM Hourly Energy Consumption, https://ieee-dataport.org/documents/pjm-hourly-energy-consumption
- [D4] GitHub: navyasancheti/PJME-Hourly-Energy-Consumption, https://github.com/navyasancheti/PJME-Hourly-Energy-Consumption
- [D5] GitHub: panambY/Hourly_Energy_Consumption, https://github.com/panambY/Hourly_Energy_Consumption
- Berkas `PJME_hourly.csv` dibaca dari repositori `panambY/Hourly_Energy_Consumption` [D5]; README-nya menyebut asal data: situs PJM dan Kaggle.
- Pedregosa dkk., *Scikit-learn: Machine Learning in Python*, JMLR 12, 2011 (K-Means, HistGradientBoosting, Random Forest, Ridge, TimeSeriesSplit).
- Library Python `holidays` (kalender libur federal AS, `holidays.US`).

## 10. Lampiran
Seluruh angka pada laporan berasal dari keluaran notebook ini (`results/results.json` untuk PJME, `results/results_regions.json` untuk 10 wilayah). Data wilayah ada di folder `data_pjm/`. Benih acak 42. Waktu eksekusi notebook sekitar 40 menit pada 2 CPU, belum termasuk grid penuh *clustering* di `cluster_tuning/` (sekitar 45 menit per wilayah untuk PJME).


### Lampiran A. Menyimpan hasil
Sel berikut menyimpan seluruh angka ke `results/results.json`.

In [27]:
json.dump(RES, open("results/results.json", "w"), indent=1, default=float); print("results/results.json tersimpan")


results/results.json tersimpan
